# Bug severity triage — context engineering experiment

Does giving Claude retrieved similar past bug reports (with their known severity) improve severity classification versus a no-context baseline? This notebook runs the full pipeline end to end: data prep -> retrieval smoke test -> baseline classification -> context-engineered classification -> evaluation.

**You need to bring your own:**
1. `data/raw/sev_train.csv`, `data/raw/sev_test.csv`, `data/raw/embedding.npy`, `data/raw/vocab.lst`
2. An `ANTHROPIC_API_KEY` (you will be prompted for it below; it is not stored in this notebook)

This mirrors how the [ticket-triage LoRA project](https://github.com/bmwelu12/ticket-triage-lora-finetuning) was actually run — the real, paid execution happens here, in Colab, with your own key and your own data, not inside a chat session.

## 0. Setup: clone the repo and install dependencies

In [ ]:
import os

REPO_DIR = "bug-severity-context-eng"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/bmwelu12/bug-severity-context-eng.git
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Upload your data

Run this cell and select `sev_train.csv`, `sev_test.csv`, `embedding.npy`, and `vocab.lst` when prompted. If you're not on Colab (e.g. running locally), just copy the four files into `data/raw/` yourself and skip this cell.

In [ ]:
import shutil

os.makedirs("data/raw", exist_ok=True)

try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        shutil.move(name, os.path.join("data/raw", name))
except ImportError:
    print("Not running on Colab — put sev_train.csv, sev_test.csv, embedding.npy, "
          "and vocab.lst into data/raw/ yourself, then re-run this cell to confirm.")

required = ["sev_train.csv", "sev_test.csv", "embedding.npy", "vocab.lst"]
missing = [f for f in required if not os.path.exists(f"data/raw/{f}")]
if missing:
    print(f"Still missing: {missing}")
else:
    print("All four data files are in place.")

## 2. Enter your Anthropic API key

Prompted, not typed into a cell, so it doesn't end up saved in the notebook or in git history.

In [ ]:
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

## 3. Data prep

Cleans `data/raw/sev_{train,test}.csv` and writes `data/processed/sev_{train,test}_clean.csv`. Prints row counts and label balance so you can sanity-check against the README (27,998 train / 4,427 test rows, ~76/24 label split).

In [ ]:
!python3 scripts/01_data_prep.py --task sev

## 4. Retrieval smoke test

Sanity-checks the embedding-based retriever on a handful of real test examples before spending API budget on the classification runs. Look at whether the retrieved neighbors are actually similar in *severity*, not just topic — this is the thing the README flags as questionable.

In [ ]:
!python3 scripts/02_retrieval.py

## 5. Classify — cheap first pass (n=300)

Runs both modes on a 300-row subsample before committing to the full 4,427-row test set. Each call costs real API spend — this step is unverified until you run it.

In [ ]:
!python3 scripts/03_classify.py --mode baseline --n 300

In [ ]:
!python3 scripts/03_classify.py --mode context --n 300 --k 3

## 6. Evaluate

Per-class precision/recall/F1 for both runs, plus the majority-class collapse check. The number that answers the actual research question is the **class-1 (severe) F1** comparison between baseline and context — not overall accuracy, since the 76/24 class split makes accuracy alone misleading.

In [ ]:
!python3 scripts/04_eval.py --task sev

## 7. Full test set (optional)

Once the n=300 pass looks sane, drop `--n` to run all 4,427 test rows through both modes (2x the API spend of the cheap pass). Re-run `04_eval.py` afterward.

In [ ]:
# !python3 scripts/03_classify.py --mode baseline
# !python3 scripts/03_classify.py --mode context --k 3
# !python3 scripts/04_eval.py --task sev

## The honest question this answers

Retrieved neighbors were topically similar but not obviously severity-similar in the retrieval smoke test (uniformly high 0.97-0.99 cosine similarity, and at least one "blocker" query pulling back a "normal" neighbor). Does that noisy-but-plausible context still move the severe-class F1, or does it just add tokens without adding signal? Whatever the eval output above shows — improvement, no change, or a regression — is the real, reportable answer; there's no result here that counts as a failure of the experiment itself.